In [5]:
# Check GPU

!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
# DocuTrace - AI Environment Setup

!pip install -q ultralytics
!pip install -q opencv-python-headless
!pip install -q albumentations
!pip install -q scikit-learn
!pip install -q matplotlib
!pip install -q seaborn
!pip install -q pandas
!pip install -q pillow
!pip install -q pytesseract
!pip install -q pdf2image

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 2.7 MB/s eta 0:00:00


In [ ]:
# Check installed libraries

import torch
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("OpenCV:", cv2.__version__)
print("Ultralytics: OK")
print("All libraries loaded successfully!")

In [ ]:
import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_f67d3d35c956e1ee3db6cf365f2da577'

In [ ]:
# Install the Kaggle API client
!pip install kaggle

# Make sure the Kaggle configuration directory exists
!mkdir -p ~/.kaggle

# Copy the kaggle.json (if you had one, but we are using os.environ, so this is not strictly needed here)
# For a fresh setup, after setting os.environ, this step is often skipped.
# However, if you have a local kaggle.json, you would copy it here:
# !cp /path/to/your/kaggle.json ~/.kaggle/

# Change permissions (if kaggle.json was copied manually)
# !chmod 600 ~/.kaggle/kaggle.json

# Example: Download a dataset (e.g., Titanic competition data)
# You can find the dataset or competition API command on Kaggle under the Data tab.
# For example, for the Titanic competition, it would be 'kaggle competitions download -c titanic'
!kaggle competitions download -c titanic

# List the downloaded files (optional)
!ls

In [ ]:
!kaggle datasets download -d dinmkeljiame/doctamper

In [ ]:
!ls -lh

In [ ]:
# Inspect DocTamper ZIP structure without extracting it

!unzip -l /content/doctamper.zip | head -100

In [ ]:
# Check the ZIP size

!du -h /content/doctamper.zip

In [ ]:
# Extract DocTamper

!mkdir -p /content/DocTamper
!unzip -q /content/doctamper.zip -d /content/DocTamper

print("Extraction completed!")

In [ ]:
import os

for root, dirs, files in os.walk("/content/DocTamper"):
    level = root.replace("/content/DocTamper", "").count(os.sep)
    if level < 2:
        print("  " * level + os.path.basename(root) + "/")

In [ ]:
!pip install -q lmdb

In [ ]:
import lmdb
import os

DATASET_PATH = "/content/DocTamper/DocTamperV1-TrainingSet"

print("Checking:", DATASET_PATH)
print("Exists:", os.path.exists(DATASET_PATH))

env = lmdb.open(
    DATASET_PATH,
    readonly=True,
    lock=False,
    readahead=False
)

with env.begin() as txn:
    stat = env.stat()
    print("Total entries:", stat["entries"])

env.close()

In [ ]:
import lmdb

DATASET_PATH = "/content/DocTamper/DocTamperV1-TrainingSet"

env = lmdb.open(
    DATASET_PATH,
    readonly=True,
    lock=False,
    readahead=False
)

with env.begin() as txn:
    cursor = txn.cursor()

    for i, (key, value) in enumerate(cursor):
        print(f"Entry {i}")
        print("Key:", key)
        print("Value size:", len(value), "bytes")
        print("-" * 50)

        if i == 9:
            break

env.close()

In [ ]:
# Reuse the already-open LMDB environment

with env.begin() as txn:
    cursor = txn.cursor()

    # Move to the first entry
    if cursor.first():
        first_key, first_value = cursor.item()

        print("Exact key:", repr(first_key))
        print("Value size:", len(first_value), "bytes")
        print("Retrieved successfully:", first_value is not None)
    else:
        print("LMDB is empty.")

In [ ]:
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

with env.begin() as txn:
    cursor = txn.cursor()

    if cursor.first():
        key, value = cursor.item()

        image = Image.open(BytesIO(value))

        print("Key:", key)
        print("Format:", image.format)
        print("Size:", image.size)
        print("Mode:", image.mode)

        plt.figure(figsize=(14, 10))
        plt.imshow(image)
        plt.axis("off")
        plt.show()

In [ ]:
# Look at all keys in the TrainingSet LMDB
# We will inspect the first 30 keys only.

with env.begin() as txn:
    cursor = txn.cursor()

    count = 0

    while cursor.next() and count < 30:
        key, value = cursor.item()

        print(
            count,
            "Key:", key,
            "| Size:", len(value), "bytes"
        )

        count += 1

In [ ]:
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
import numpy as np

with env.begin() as txn:

    # First image
    image_bytes = txn.get(b"image-000000000")

    # Corresponding label/mask
    label_bytes = txn.get(b"label-000000000")

    print("Image found:", image_bytes is not None)
    print("Label found:", label_bytes is not None)

    if image_bytes and label_bytes:

        image = Image.open(BytesIO(image_bytes))
        mask = Image.open(BytesIO(label_bytes))

        print("Image:", image.size, image.mode)
        print("Mask :", mask.size, mask.mode)

        # Display both
        plt.figure(figsize=(14, 6))

        plt.subplot(1, 2, 1)
        plt.imshow(image)
        plt.title("Document Image")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(mask, cmap="gray")
        plt.title("Tampering Mask")
        plt.axis("off")

        plt.show()

In [ ]:
import numpy as np

with env.begin() as txn:
    label_bytes = txn.get(b"label-000000000")

mask = Image.open(BytesIO(label_bytes)).convert("L")
mask_array = np.array(mask)

print("Mask shape:", mask_array.shape)
print("Minimum value:", mask_array.min())
print("Maximum value:", mask_array.max())

unique_values = np.unique(mask_array)

print("Number of unique values:", len(unique_values))
print("First values:", unique_values[:20])

In [ ]:
import numpy as np
from PIL import Image
from io import BytesIO

sample_size = 1000

tampered_count = 0
authentic_count = 0
missing_count = 0

with env.begin() as txn:

    for i in range(sample_size):

        # DocTamper labels use 9 digits
        key = f"label-{i:09d}".encode()

        label_bytes = txn.get(key)

        if label_bytes is None:
            missing_count += 1
            continue

        mask = np.array(
            Image.open(BytesIO(label_bytes)).convert("L")
        )

        if np.any(mask == 1):
            tampered_count += 1
        else:
            authentic_count += 1

total = tampered_count + authentic_count

print("Sample checked:", sample_size)
print("Tampered:", tampered_count)
print("Authentic:", authentic_count)
print("Missing:", missing_count)

if total > 0:
    print("\nTampered %:", round(tampered_count / total * 100, 2))
    print("Authentic %:", round(authentic_count / total * 100, 2))

In [ ]:
# Check DocTamper official TestingSet

import lmdb

TEST_PATH = "/content/DocTamper/DocTamperV1-TestingSet"

test_env = lmdb.open(
    TEST_PATH,
    readonly=True,
    lock=False,
    readahead=False
)

with test_env.begin() as txn:
    stat = txn.stat()
    print("TestingSet entries:", stat["entries"])

test_env.close()

In [ ]:
import lmdb
import numpy as np
from PIL import Image
from io import BytesIO

TEST_PATH = "/content/DocTamper/DocTamperV1-TestingSet"

# Reopen TestingSet
test_env = lmdb.open(
    TEST_PATH,
    readonly=True,
    lock=False,
    readahead=False
)

tampered_count = 0
authentic_count = 0
missing_count = 0

with test_env.begin() as txn:

    for i in range(1000):

        key = f"label-{i:09d}".encode()
        label_bytes = txn.get(key)

        if label_bytes is None:
            missing_count += 1
            continue

        mask = np.array(
            Image.open(BytesIO(label_bytes)).convert("L")
        )

        if np.any(mask == 1):
            tampered_count += 1
        else:
            authentic_count += 1

total = tampered_count + authentic_count

print("Sample checked:", 1000)
print("Tampered:", tampered_count)
print("Authentic:", authentic_count)
print("Missing:", missing_count)

if total > 0:
    print("\nTampered %:", round(tampered_count / total * 100, 2))
    print("Authentic %:", round(authentic_count / total * 100, 2))

In [ ]:
# Step 15 — Create train/validation indices

import random

TOTAL_TRAINING = 240000

indices = list(range(TOTAL_TRAINING))

random.seed(42)
random.shuffle(indices)

train_indices = indices[:20000]
val_indices = indices[20000:24000]

print("Training samples:", len(train_indices))
print("Validation samples:", len(val_indices))

print("\nFirst 10 training indices:")
print(train_indices[:10])

print("\nFirst 10 validation indices:")
print(val_indices[:10])

In [ ]:
# Inspect the actual format of several label entries

with env.begin() as txn:

    for i in range(10):

        key = f"label-{i:09d}".encode()
        label_bytes = txn.get(key)

        print(f"\n{key}")

        if label_bytes is None:
            print("  NOT FOUND")
            continue

        print("  Size:", len(label_bytes), "bytes")
        print("  First 20 bytes:", label_bytes[:20])

        try:
            img = Image.open(BytesIO(label_bytes))
            print("  PIL image:", img.format, img.size, img.mode)
        except Exception as e:
            print("  NOT a PIL image")
            print("  Error:", type(e).__name__)

In [ ]:
# Find any problematic label entries in the first 1000

problematic = []

with env.begin() as txn:

    for i in range(1000):

        key = f"label-{i:09d}".encode()
        label_bytes = txn.get(key)

        if label_bytes is None:
            problematic.append((i, "missing"))
            continue

        try:
            mask = Image.open(BytesIO(label_bytes))
            mask.load()

        except Exception as e:
            problematic.append(
                (i, type(e).__name__, len(label_bytes))
            )

print("Problematic entries:", len(problematic))

for item in problematic[:20]:
    print(item)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from io import BytesIO
import numpy as np


class DocTamperDataset(Dataset):

    def __init__(self, env, indices, image_size=256):
        self.env = env
        self.indices = indices
        self.image_size = image_size

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):

        image_index = self.indices[idx]

        image_key = f"image-{image_index:09d}".encode()
        label_key = f"label-{image_index:09d}".encode()

        with self.env.begin() as txn:

            image_bytes = txn.get(image_key)
            label_bytes = txn.get(label_key)

        if image_bytes is None or label_bytes is None:
            raise RuntimeError(
                f"Missing data for index {image_index}"
            )

        # Image
        image = Image.open(
            BytesIO(image_bytes)
        ).convert("RGB")

        # Mask
        mask = Image.open(
            BytesIO(label_bytes)
        ).convert("L")

        # Resize
        image = image.resize(
            (self.image_size, self.image_size),
            Image.Resampling.BILINEAR
        )

        mask = mask.resize(
            (self.image_size, self.image_size),
            Image.Resampling.NEAREST
        )

        # NumPy
        image = np.array(
            image,
            dtype=np.float32
        ) / 255.0

        mask = np.array(
            mask,
            dtype=np.float32
        )

        # Binary mask
        mask = (mask > 0).astype(np.float32)

        # HWC → CHW
        image = torch.from_numpy(
            image.transpose(2, 0, 1)
        )

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0)

        return image, mask

In [ ]:
TRAIN_PATH = "/content/DocTamper/DocTamperV1-TrainingSet"

train_dataset = DocTamperDataset(
    env,
    train_indices,
    image_size=256
)

val_dataset = DocTamperDataset(
    env,
    val_indices,
    image_size=256
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
import lmdb
import torch
from torch.utils.data import DataLoader

# Create a fresh TrainingSet connection
TRAIN_PATH = "/content/DocTamper/DocTamperV1-TrainingSet"

train_env = lmdb.open(
    TRAIN_PATH,
    readonly=True,
    lock=False,
    readahead=False,
    max_readers=128
)

print("Training LMDB opened successfully")

# Recreate datasets using the NEW connection
train_dataset = DocTamperDataset(
    train_env,
    train_indices,
    image_size=256
)

val_dataset = DocTamperDataset(
    train_env,
    val_indices,
    image_size=256
)

# Recreate DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
# Inspect the actual LMDB keys

with train_env.begin() as txn:
    cursor = txn.cursor()

    image_count = 0
    label_count = 0
    other_count = 0

    first_images = []
    first_labels = []

    for key, value in cursor:

        key_str = key.decode()

        if key_str.startswith("image-"):
            image_count += 1

            if len(first_images) < 10:
                first_images.append(key_str)

        elif key_str.startswith("label-"):
            label_count += 1

            if len(first_labels) < 10:
                first_labels.append(key_str)

        else:
            other_count += 1

print("Image entries:", image_count)
print("Label entries:", label_count)
print("Other entries:", other_count)

print("\nFirst image keys:")
print(first_images)

print("\nFirst label keys:")
print(first_labels)

In [ ]:
import random

# 120,000 valid image-label pairs
num_samples = 120000

indices = list(range(num_samples))

# Reproducible shuffle
random.seed(42)
random.shuffle(indices)

# 90% training, 10% validation
split = int(0.9 * num_samples)

train_indices = indices[:split]
val_indices = indices[split:]

print("Total:", len(indices))
print("Training:", len(train_indices))
print("Validation:", len(val_indices))

In [ ]:
train_dataset = DocTamperDataset(
    train_env,
    train_indices,
    image_size=256
)

val_dataset = DocTamperDataset(
    train_env,
    val_indices,
    image_size=256
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
images, masks = next(iter(train_loader))

print("Images shape:", images.shape)
print("Masks shape:", masks.shape)
print("Image range:", images.min().item(), "to", images.max().item())
print("Mask values:", torch.unique(masks))

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):

    def __init__(self):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(3, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = DoubleConv(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        # Output
        self.output = nn.Conv2d(64, 1, 1)

    def forward(self, x):

        # Encoder
        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool(e1)
        )

        e3 = self.enc3(
            self.pool(e2)
        )

        e4 = self.enc4(
            self.pool(e3)
        )

        # Bottleneck
        b = self.bottleneck(
            self.pool(e4)
        )

        # Decoder
        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.output(d1)

In [ ]:
model = UNet().to(device)

print(model)

In [ ]:
images, masks = next(iter(train_loader))

images = images.to(device)
masks = masks.to(device)

with torch.no_grad():
    outputs = model(images)

print("Input shape :", images.shape)
print("Mask shape  :", masks.shape)
print("Output shape:", outputs.shape)

In [ ]:
import torch
import torch.nn as nn


class DiceLoss(nn.Module):

    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):

        probs = torch.sigmoid(logits)

        probs = probs.view(-1)
        targets = targets.view(-1)

        intersection = (probs * targets).sum()

        dice = (
            (2 * intersection + self.smooth)
            /
            (probs.sum() + targets.sum() + self.smooth)
        )

        return 1 - dice

In [ ]:
bce_loss = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss()


def combined_loss(logits, targets):

    bce = bce_loss(logits, targets)
    dice = dice_loss(logits, targets)

    return 0.5 * bce + 0.5 * dice

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

print("Optimizer:", optimizer)
print("Learning rate:", 1e-4)

In [ ]:
images, masks = next(iter(train_loader))

images = images.to(device)
masks = masks.to(device)

outputs = model(images)

loss = combined_loss(outputs, masks)

print("Output shape:", outputs.shape)
print("Loss:", loss.item())

In [ ]:
optimizer.zero_grad()

loss.backward()

optimizer.step()

print("Optimizer step successful!")

In [ ]:
import time

model.train()

test_batches = 100
running_loss = 0.0

start_time = time.time()

for batch_idx, (images, masks) in enumerate(train_loader):

    if batch_idx >= test_batches:
        break

    images = images.to(device)
    masks = masks.to(device)

    optimizer.zero_grad()

    outputs = model(images)

    loss = combined_loss(outputs, masks)

    loss.backward()

    optimizer.step()

    running_loss += loss.item()

    if (batch_idx + 1) % 10 == 0:
        print(
            f"Batch [{batch_idx + 1}/{test_batches}] "
            f"Loss: {loss.item():.4f}"
        )

elapsed = time.time() - start_time

print("\n" + "=" * 40)
print("100-BATCH TEST COMPLETE")
print("=" * 40)
print("Average Loss:", running_loss / test_batches)
print("Time:", round(elapsed / 60, 2), "minutes")

In [ ]:
model.eval()

total_loss = 0.0
total_dice = 0.0
total_iou = 0.0

num_batches = 50  # quick validation test

with torch.no_grad():

    for batch_idx, (images, masks) in enumerate(val_loader):

        if batch_idx >= num_batches:
            break

        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)

        loss = combined_loss(outputs, masks)
        total_loss += loss.item()

        # Convert logits to binary prediction
        predictions = (torch.sigmoid(outputs) > 0.5).float()

        # Dice
        intersection = (predictions * masks).sum()
        dice = (
            (2 * intersection + 1)
            /
            (predictions.sum() + masks.sum() + 1)
        )

        # IoU
        union = (
            predictions.sum()
            + masks.sum()
            - intersection
        )

        iou = (intersection + 1) / (union + 1)

        total_dice += dice.item()
        total_iou += iou.item()


print("=" * 40)
print("VALIDATION RESULTS")
print("=" * 40)

print("Validation Loss:",
      round(total_loss / num_batches, 4))

print("Dice Score:",
      round(total_dice / num_batches, 4))

print("IoU Score:",
      round(total_iou / num_batches, 4))

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": total_loss / num_batches
}, "/content/doctamper_unet_test.pth")

print("Test checkpoint saved successfully!")

In [ ]:
def calculate_metrics(predictions, targets):

    predictions = (torch.sigmoid(predictions) > 0.5).float()

    intersection = (predictions * targets).sum()

    dice = (
        (2 * intersection + 1e-6)
        /
        (predictions.sum() + targets.sum() + 1e-6)
    )

    union = (
        predictions.sum()
        + targets.sum()
        - intersection
    )

    iou = (
        (intersection + 1e-6)
        /
        (union + 1e-6)
    )

    return dice.item(), iou.item()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

CHECKPOINT_DIR = "/content/drive/MyDrive/DocuTrace_Checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Checkpoint directory:")
print(CHECKPOINT_DIR)

In [ ]:
import os
import time
import torch
import json

# ============================================================
# CONFIGURATION
# ============================================================

EPOCHS = 3

CHECKPOINT_DIR = "/content/drive/MyDrive/DocuTrace_Checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

LATEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "latest_checkpoint.pth"
)

BEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best_model.pth"
)

HISTORY_FILE = os.path.join(
    CHECKPOINT_DIR,
    "training_history.json"
)

print("Checkpoint directory:")
print(CHECKPOINT_DIR)


# ============================================================
# METRICS
# ============================================================

def calculate_metrics(predictions, targets):

    predictions = (
        torch.sigmoid(predictions) > 0.5
    ).float()

    intersection = (
        predictions * targets
    ).sum()

    dice = (
        (2 * intersection + 1e-6)
        /
        (
            predictions.sum()
            + targets.sum()
            + 1e-6
        )
    )

    union = (
        predictions.sum()
        + targets.sum()
        - intersection
    )

    iou = (
        (intersection + 1e-6)
        /
        (union + 1e-6)
    )

    return dice.item(), iou.item()


# ============================================================
# LOAD PREVIOUS CHECKPOINT IF AVAILABLE
# ============================================================

start_epoch = 0
best_val_loss = float("inf")

history = {
    "train_loss": [],
    "val_loss": [],
    "val_dice": [],
    "val_iou": []
}


if os.path.exists(LATEST_CHECKPOINT):

    print("\nPrevious checkpoint found!")
    print("Loading checkpoint...")

    checkpoint = torch.load(
        LATEST_CHECKPOINT,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    start_epoch = checkpoint["epoch"]

    best_val_loss = checkpoint[
        "best_val_loss"
    ]

    history = checkpoint["history"]

    print(
        f"Resuming from Epoch {start_epoch}"
    )

    print(
        f"Best validation loss: "
        f"{best_val_loss:.4f}"
    )

else:

    print("\nNo previous checkpoint found.")
    print("Starting training from Epoch 1.")


# ============================================================
# TRAINING
# ============================================================

for epoch in range(start_epoch, EPOCHS):

    print("\n")
    print("=" * 70)
    print(
        f"EPOCH {epoch + 1}/{EPOCHS}"
    )
    print("=" * 70)

    start_time = time.time()


    # ========================================================
    # TRAIN
    # ========================================================

    model.train()

    train_loss = 0.0

    for batch_idx, (images, masks) in enumerate(
        train_loader
    ):

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        outputs = model(images)

        loss = combined_loss(
            outputs,
            masks
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()


        # Progress every 500 batches

        if (batch_idx + 1) % 500 == 0:

            current_loss = (
                train_loss
                / (batch_idx + 1)
            )

            print(
                f"Batch "
                f"{batch_idx + 1}/"
                f"{len(train_loader)} "
                f"| Loss: "
                f"{loss.item():.4f} "
                f"| Avg: "
                f"{current_loss:.4f}"
            )


    train_loss /= len(train_loader)


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_loss = 0.0
    val_dice = 0.0
    val_iou = 0.0

    with torch.no_grad():

        for images, masks in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            masks = masks.to(
                device,
                non_blocking=True
            )

            outputs = model(images)

            loss = combined_loss(
                outputs,
                masks
            )

            val_loss += loss.item()

            dice, iou = calculate_metrics(
                outputs,
                masks
            )

            val_dice += dice
            val_iou += iou


    val_loss /= len(val_loader)
    val_dice /= len(val_loader)
    val_iou /= len(val_loader)


    # ========================================================
    # SAVE HISTORY
    # ========================================================

    history["train_loss"].append(
        train_loss
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_dice"].append(
        val_dice
    )

    history["val_iou"].append(
        val_iou
    )


    # ========================================================
    # TIME
    # ========================================================

    elapsed = (
        time.time() - start_time
    )

    print("\n")
    print("-" * 50)
    print("EPOCH RESULTS")
    print("-" * 50)

    print(
        f"Train Loss : {train_loss:.4f}"
    )

    print(
        f"Val Loss   : {val_loss:.4f}"
    )

    print(
        f"Val Dice   : {val_dice:.4f}"
    )

    print(
        f"Val IoU    : {val_iou:.4f}"
    )

    print(
        f"Time       : "
        f"{elapsed / 60:.2f} minutes"
    )


    # ========================================================
    # SAVE LATEST CHECKPOINT
    # ========================================================

    checkpoint = {

        "epoch": epoch + 1,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "best_val_loss":
            best_val_loss,

        "history":
            history,

        "train_loss":
            train_loss,

        "val_loss":
            val_loss,

        "val_dice":
            val_dice,

        "val_iou":
            val_iou
    }


    torch.save(
        checkpoint,
        LATEST_CHECKPOINT
    )

    print(
        "\n✓ Latest checkpoint saved!"
    )


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        checkpoint[
            "best_val_loss"
        ] = best_val_loss

        torch.save(
            checkpoint,
            BEST_CHECKPOINT
        )

        print(
            "✓ New BEST model saved!"
        )


    # ========================================================
    # SAVE HISTORY JSON
    # ========================================================

    with open(
        HISTORY_FILE,
        "w"
    ) as f:

        json.dump(
            history,
            f,
            indent=4
        )

    print(
        "✓ Training history saved!"
    )


# ============================================================
# TRAINING COMPLETE
# ============================================================

print("\n")
print("=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    f"Best Validation Loss: "
    f"{best_val_loss:.4f}"
)

print("\nFiles saved in:")
print(CHECKPOINT_DIR)